# 三谋网关联调操作记录

2026-10-27，源码 a1ab51e。此文件保存本轮启动与验收代码，便于查阅和恢复；包含分阶段执行与一次失败诊断，不应直接全部运行。可重复重建请使用仓库 scripts/model-serving/colab_gateway_bootstrap.py 和 docs/runbooks/live-gateway-colab.md。

上传单元格使用本轮发送前的本地原件；其余单元格从当前Colab读取。不包含执行输出或密钥值。

In [ ]:
print("Hello Colab!")

## 三谋项目网关实测 · 2026-10-27

固定源码：`a1ab51e5ab3df30fd3da749b3e52502053f4dfae`。从仓库恢复可重复启动包，保留上方原始 Hello 单元格。

流程：硬件检查 → 导入源码包 → 独立 Python 3.12 环境和固定模型 → 项目网关两轮工具往返。模型只提出调用，工具返回固定测试数据，不操作游戏。测试原件会保存到 Mac 项目。


In [ ]:
import subprocess,sys,shutil,json
print(subprocess.check_output(['nvidia-smi','--query-gpu=name,memory.total,memory.used,driver_version','--format=csv,noheader'],text=True))
print('GPU_PROCESSES',subprocess.check_output(['nvidia-smi','--query-compute-apps=pid,used_memory','--format=csv,noheader'],text=True))
print(json.dumps({'python':sys.version,'python312':shutil.which('python3.12'),'free_gib':round(shutil.disk_usage('/content').free/2**30,1)}))

In [ ]:
import base64,hashlib,json,pathlib,tempfile,os,sys,subprocess
upload_dir=pathlib.Path(tempfile.mkdtemp(prefix='sanmou-gateway-upload-',dir='/content'))
upload_payloads={'source.tar.gz': '...', 'source-inventory.json': '...', 'bundle-manifest.json': 'ewogICJhcmNoaXZlIjogewogICAgImJ5dGVzIjogNDE0NjI2LAogICAgInBhdGgiOiAic291cmNlLnRhci5neiIsCiAgICAic2hhMjU2IjogImE2MmY4ZmNiNDAzMzQyMzhhMmU4YjkxZmY5NmMwYTZjN2QxZTdjYmVlZmI0NDhiMmFlZmIxODcxZTM4ZjBhMTkiCiAgfSwKICAiYm9vdHN0cmFwIjogewogICAgImJ5dGVzIjogMzExNTksCiAgICAicGF0aCI6ICJjb2xhYl9nYXRld2F5X2Jvb3RzdHJhcC5weSIsCiAgICAic2hhMjU2IjogImQxMDY1ZTNkZDcxY2MyMWY3NTZiZTgyYTg3MTAxOTY3NTFkYmUwNjk2NGIzMmEwYTlmMDRjNjg2YmFmMzlkOWYiCiAgfSwKICAibW9kZWxfZmlsZXNfcGF0aCI6ICJkb2NzL3Rlc3QtcmVwb3J0cy8yMDI2LTEwLTI0L2NvbGFiLW5hdGl2ZS9tb2RlbC1maWxlcy5qc29uIiwKICAibW9kZWxfaWQiOiAiUXdlbi9Rd2VuMy41LTlCIiwKICAibW9kZWxfcmV2aXNpb24iOiAiYzIwMjIzNjIzNTc2MmUxYzg3MWFkMGNjYjYwYzhlZTViYTMzN2I5YSIsCiAgInJlcXVpcmVtZW50c19wYXRoIjogImRvY3MvdGVzdC1yZXBvcnRzLzIwMjYtMTAtMjQvY29sYWItbmF0aXZlL3JlcXVpcmVtZW50cy1weTMxMi5sb2NrIiwKICAic2NoZW1hIjogInNhbm1vdS1jb2xhYi1nYXRld2F5LWJ1bmRsZS12MSIsCiAgInNvdXJjZV9jb21taXQiOiAiYTFhYjUxZTVhYjNkZjMwZmQzZGE3NDliM2U1MjUwMjA1M2Y0ZGZhZSIsCiAgInNvdXJjZV9pbnZlbnRvcnkiOiB7CiAgICAiYnl0ZXMiOiAzNzk1MCwKICAgICJwYXRoIjogInNvdXJjZS1pbnZlbnRvcnkuanNvbiIsCiAgICAic2hhMjU2IjogIjE5YWNlMjk4OTAwNWNjNmEzMTZjOGYzODY1MWNmNjVmMmNmOTU3MmE0NTRlNzUzYmFhNmY0NTQ2NWJiN2NhNDciCiAgfQp9Cg==', 'bundle-manifest.sha256': 'NmM4NzQ3YzNkM2RlMjBmMmQ1NTI3NzI4Y2ZkMWE2ZGM3MDJkNjQwMzU0MzY0YWFkM2NjNjBhM2MxNzM1ZmFmZQo=', 'colab_gateway_bootstrap.py': '...'}
upload_hashes={'source.tar.gz': 'a62f8fcb40334238a2e8b91ff96c0a6c7d1e7cbeefb448b2aefb1871e38f0a19', 'source-inventory.json': '19ace2989005cc6a316c8f38651cf65f2cf9572a454e753baa6f45465bb7ca47', 'bundle-manifest.json': '6c8747c3d3de20f2d5527728cfd1a6dc702d640354364aad3cc60a3c1735fafe', 'bundle-manifest.sha256': '473d627d8918433437681f9aceb30b17e21f30f6ec7158051f1ac70b01000e32', 'colab_gateway_bootstrap.py': 'd1065e3dd71cc21f756be82a8710196751dbe06964b32a0a9f04c686baf39d9f'}
for name,value in upload_payloads.items():
 b=base64.b64decode(value,validate=True);assert hashlib.sha256(b).hexdigest()==upload_hashes[name]
 with (upload_dir/name).open('xb') as f:f.write(b)
 os.chmod(upload_dir/name,0o600)
print(json.dumps({'upload_dir':str(upload_dir),'verified_files':len(upload_payloads)}))


In [ ]:
bootstrap_log=(upload_dir/'bootstrap-run.log').open('x')
bootstrap_cmd=[sys.executable,str(upload_dir/'colab_gateway_bootstrap.py'),'--bundle-dir',str(upload_dir),'--mode','provision']
bootstrap_proc=subprocess.Popen(bootstrap_cmd,stdout=bootstrap_log,stderr=subprocess.STDOUT,start_new_session=True)
print('BOOTSTRAP_PID',bootstrap_proc.pid)

In [ ]:
print('Gateway live test passed: 2 rounds, 1088 tokens. Task model server stopped. Evidence archived to Mac. Rebuild from the committed scripts; do not run all historical cells together.')

In [ ]:
control_dir=pathlib.Path(tempfile.mkdtemp(prefix='sanmou-gateway-control-',dir='/content'))
clean_bundle=control_dir/'bundle';clean_bundle.mkdir(mode=0o700)
for name in upload_payloads:
 b=(upload_dir/name).read_bytes();assert hashlib.sha256(b).hexdigest()==upload_hashes[name]
 with (clean_bundle/name).open('xb') as f:f.write(b)
 os.chmod(clean_bundle/name,0o600)
bootstrap_log=(control_dir/'bootstrap-run.log').open('x')
bootstrap_cmd=[sys.executable,str(clean_bundle/'colab_gateway_bootstrap.py'),'--bundle-dir',str(clean_bundle),'--mode','provision']
bootstrap_proc=subprocess.Popen(bootstrap_cmd,stdout=bootstrap_log,stderr=subprocess.STDOUT,env=dict(os.environ,PYTHONDONTWRITEBYTECODE='1'),start_new_session=True)
print('CONTROL',str(control_dir),'BOOTSTRAP_PID',bootstrap_proc.pid)

In [ ]:
assert bootstrap_proc.poll() == 0, 'provision has not completed successfully'
import json,pathlib,secrets,socket,os,subprocess,sys,time,hashlib
gw_task=pathlib.Path('/content/sanmou-gateway-5p9xpxp4')
provision=json.loads((gw_task/'provision-status.json').read_text())
assert provision['status']=='provisioned'
source_root=pathlib.Path(provision['source_root'])
sg_python=provision['python']
model_dir=pathlib.Path(provision['model_dir'])
with socket.socket() as sock:
    sock.bind(('127.0.0.1',0));native_port=sock.getsockname()[1]
public_config={
 'model-path':str(model_dir),'tokenizer-path':str(model_dir),
 'chat-template':str(model_dir/'chat_template.jinja'),
 'served-model-name':'sanmou-qwen35-9b-gateway','host':'127.0.0.1','port':native_port,
 'dtype':'bfloat16','load-format':'safetensors','tensor-parallel-size':1,'data-parallel-size':1,
 'context-length':8192,'mem-fraction-static':0.55,'max-running-requests':1,'chunked-prefill-size':1024,
 'cuda-graph-backend-decode':'disabled','cuda-graph-backend-prefill':'disabled',
 'tool-call-parser':'qwen3_coder','reasoning-parser':'qwen3','log-level':'warning',
 'log-requests':False,'trust-remote-code':False,
 'limit-mm-data-per-request':{'image':0,'video':0,'audio':0},'skip-server-warmup':True,
}
with (gw_task/'public-runtime.json').open('x') as f:json.dump(public_config,f,indent=2)
service_key=secrets.token_urlsafe(32);admin_key=secrets.token_urlsafe(32);gateway_key=secrets.token_urlsafe(32)
fd=os.open(gw_task/'private-runtime.yaml',os.O_CREAT|os.O_EXCL|os.O_WRONLY,0o600)
with os.fdopen(fd,'w') as f:json.dump(dict(public_config,**{'api-key':service_key,'admin-api-key':admin_key}),f)
run_env=boot.isolated_env(gw_task,source_root)
run_env.update(HF_HUB_OFFLINE='1',TRANSFORMERS_OFFLINE='1',PYTHONDONTWRITEBYTECODE='1')
scenario_run=subprocess.run([sg_python,str(source_root/'scripts/model-serving/live_gateway_roundtrip.py'),'--describe-scenario'],env=run_env,capture_output=True,text=True,timeout=30)
assert scenario_run.returncode==0, 'scenario description failed'
scenario=json.loads(scenario_run.stdout)
assert scenario['prompt_sha256']=='725ca696214f9e2c924582772b434f7a88e9f5cad2e68f63f8fca46d6dd28674'
with (gw_task/'scenario-description.json').open('x') as f:f.write(scenario_run.stdout)
with (gw_task/'sampling.json').open('x') as f:json.dump(scenario['sampling'],f)
binding_cmd=[sg_python,str(source_root/'scripts/model-serving/native_runtime_manifest.py'),
 '--model-dir',str(model_dir),'--runtime-config',str(gw_task/'public-runtime.json'),
 '--expected-model-files',provision['expected_model_files'],'--sampling',str(gw_task/'sampling.json'),
 '--prompt-sha256',scenario['prompt_sha256'],'--source-root',str(source_root),
 '--source-inventory',provision['source_inventory'],'--output',str(gw_task/'native-binding')]
binding_log=(gw_task/'binding-run.log').open('x')
binding_proc=subprocess.Popen(binding_cmd,env=run_env,stdout=binding_log,stderr=subprocess.STDOUT)
print(json.dumps({'binding_pid':binding_proc.pid,'work_dir':str(gw_task),'native_port':native_port,'source_commit':provision['source_commit']}))


In [ ]:
assert binding_proc.poll()==0
assert not subprocess.check_output(['nvidia-smi','--query-compute-apps=pid','--format=csv,noheader'],text=True).strip()
server_log=(gw_task/'server.log').open('x');server_start=time.monotonic()
server_cmd=['timeout','1200',sg_python,'-m','sglang.launch_server','--config',str(gw_task/'private-runtime.yaml')]
server_proc=subprocess.Popen(server_cmd,env=run_env,stdout=server_log,stderr=subprocess.STDOUT,start_new_session=True)
with (gw_task/'launch-record.json').open('x') as f:json.dump({'pid':server_proc.pid,'public_config_sha256':hashlib.sha256((gw_task/'public-runtime.json').read_bytes()).hexdigest(),'source_commit':provision['source_commit'],'service_deadline_seconds':1200},f)
print('SERVER_PID',server_proc.pid)

In [ ]:
warmup=dict(scenario['sampling'],model=public_config['served-model-name'],messages=[{'role':'user','content':'Reply with exactly READY.'}],max_tokens=32,n=1,stream=False,logprobs=True,return_token_ids=True,return_meta_info=True,chat_template_kwargs={'enable_thinking':False})
body=json.dumps(warmup,separators=(',',':')).encode()
with (gw_task/'warmup-request.json').open('xb') as f:f.write(body)
t=time.monotonic();r=http.post(f'http://127.0.0.1:{native_port}/v1/chat/completions',data=body,headers={'Authorization':'Bearer '+service_key,'Content-Type':'application/json'},timeout=180)
with (gw_task/'warmup-response.json').open('xb') as f:f.write(r.content)
with (gw_task/'warmup-timing.json').open('x') as f:json.dump({'status':r.status_code,'seconds':time.monotonic()-t},f)
print('WARMUP',r.status_code,r.json().get('usage'),r.json().get('choices',[{}])[0].get('message'))

In [ ]:
assert r.status_code==200 and r.json()['choices'][0]['message']['content']=='READY'
runner_env=dict(run_env,POLICY_GATEWAY_API_KEY=gateway_key,SGLANG_API_KEY=service_key)
runner_cmd=[sg_python,str(source_root/'scripts/model-serving/live_gateway_roundtrip.py'),'--manifest',str(gw_task/'native-binding/policy-manifest.json'),'--binding',str(gw_task/'native-binding/runtime-receipt.json'),'--upstream',f'http://127.0.0.1:{native_port}','--output',str(gw_task/'gateway-evidence')]
runner_log=(gw_task/'gateway-run.log').open('x');runner_started=time.monotonic()
runner_proc=subprocess.Popen(runner_cmd,env=runner_env,stdout=runner_log,stderr=subprocess.STDOUT,start_new_session=True)
print('RUNNER_PID',runner_proc.pid)

In [ ]:
import signal
assert runner_proc.poll()==0
with (gw_task/'gpu-after-requests.txt').open('x') as f:f.write(subprocess.check_output(['nvidia-smi','--query-gpu=name,memory.total,memory.used,driver_version','--format=csv,noheader'],text=True))
shutdown={'owned_process_pid':server_proc.pid,'requested_signal':'SIGTERM','runtime_stopped':False}
if server_proc.poll() is None:os.killpg(server_proc.pid,signal.SIGTERM)
try:shutdown['exit_code']=server_proc.wait(timeout=20)
except subprocess.TimeoutExpired:shutdown['exit_code']='timeout'
shutdown['gpu_processes_after']=subprocess.check_output(['nvidia-smi','--query-compute-apps=pid,used_memory','--format=csv,noheader'],text=True).strip()
with (gw_task/'shutdown.json').open('x') as f:json.dump(shutdown,f,indent=2)
print(json.dumps(shutdown))

In [ ]:
assert runner_proc.poll()==0 and server_proc.poll() is not None
import datetime,gzip,base64,hashlib,json,os,stat
secrets_to_exclude=(service_key,admin_key,gateway_key)
pub=json.loads((gw_task/'public-runtime.json').read_text())
private=json.loads((gw_task/'private-runtime.yaml').read_text())
runtime_checks={
 'collected_at_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),
 'source_commit':provision['source_commit'],
 'public_private_nonsecret_config_match':{k:v for k,v in private.items() if k not in ('api-key','admin-api-key')}==pub,
 'private_config_mode':oct(stat.S_IMODE((gw_task/'private-runtime.yaml').stat().st_mode)),
 'warmup_model_calls':1,'gateway_model_calls':summary['native_http_dispatches'],
 'total_model_calls':1+summary['native_http_dispatches'],'game_input_dispatches':0,
 'runtime_created_or_stopped':False,'model_service_stopped':server_proc.poll() is not None,
 'bootstrap_initial_failure':'log file placed in strict five-file bundle; retry used clean bundle and separate log directory',
}
with (gw_task/'runtime-checks.json').open('x') as f:json.dump(runtime_checks,f,indent=2)
export_data={}
for folder in ('gateway-evidence','native-binding'):
 for p in sorted((gw_task/folder).iterdir()):
  assert p.is_file() and not p.is_symlink()
  export_data[folder+'/'+p.name]=p.read_bytes()
names=['provision-status.json','public-runtime.json','launch-record.json','scenario-description.json','sampling.json',
 'warmup-request.json','warmup-response.json','warmup-timing.json','runtime-versions.log','runtime-versions.json',
 'pip-check.log','pip-check.json','gateway-import-readiness.log','gateway-import-readiness.json',
 'download-model.json','install-lock.json','bootstrap-pip.json','create-venv.json','binding-run.log','gateway-run.log',
 'gpu-after-requests.txt','shutdown.json','runtime-checks.json']
for name in names:export_data[name]=(gw_task/name).read_bytes()
for key,p in [('server.log',gw_task/'server.log'),('bootstrap-initial-failure.log',upload_dir/'bootstrap-run.log')]:
 text=p.read_text()
 for value in secrets_to_exclude:text=text.replace(value,'[REDACTED]')
 export_data[key]=text.encode()
assert all(value.encode() not in raw for raw in export_data.values() for value in secrets_to_exclude)
outer_index={'schema':'sanmou-live-gateway-receipts-v1','files':{n:{'bytes':len(b),'sha256':hashlib.sha256(b).hexdigest()} for n,b in sorted(export_data.items())}}
export_data['evidence-index.json']=json.dumps(outer_index,indent=2).encode()
transfer_raw=gzip.compress(json.dumps({n:base64.b64encode(b).decode() for n,b in export_data.items()},sort_keys=True).encode(),mtime=0)
transfer_b64=base64.b64encode(transfer_raw).decode()
print(json.dumps({'files':len(outer_index['files']),'bytes':sum(x['bytes'] for x in outer_index['files'].values()),'transfer_chars':len(transfer_b64),'transfer_sha256':hashlib.sha256(transfer_raw).hexdigest()}))


## 2026-10-27 网关联调完成

- 项目网关到真实 Qwen3.5-9B 两轮通过，共 1088 token。
- 无密钥、错密钥和错版本返回 401、401、409，没有增加模型调用。
- 源码版本：`a1ab51e5ab3df30fd3da749b3e52502053f4dfae`。
- 本页保留操作过程；后续重建请使用仓库启动脚本，不要直接全部运行。
- 原始证据已读回 Mac，模型服务已停止。